In [0]:
%sql
CREATE CATALOG IF NOT EXISTS bixi_mobility;
USE CATALOG bixi_mobility;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

-- Volume pour stocker les fichiers bruts avant ingestion
CREATE VOLUME IF NOT EXISTS bronze.landing_zone;
-- Création du volume qui accueillera les checkpoints de streaming du Silver
CREATE VOLUME IF NOT EXISTS bixi_mobility.silver.checkpoints;

In [0]:
#config des chemins

catalog = "bixi_mobility"
landing_path = f"/Volumes/{catalog}/bronze/landing_zone"
schema_location = f"/Volumes/{catalog}/bronze/landing_zone/_schema"
checkpoint_location = f"/Volumes/{catalog}/bronze/landing_zone/_checkpoint"

target_table = f"{catalog}.bronze.trips_raw"

In [0]:
#lecture avec Auto Loader

df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
)

In [0]:
# Définition des chemins et noms de tables pour la couche Bronze

from pyspark.sql.functions import current_timestamp, col

df_bronze_enriched = (
    df_bronze
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col ("_metadata.file_path"))
)

(
    df_bronze_enriched.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)   # traitement batch unique, pas de stream 24/7
    .toTable(target_table)
)

In [0]:
%sql
-- Vérification des données dans la table Delta
SELECT COUNT(*) FROM bixi_mobility.bronze.trips_raw;

In [0]:
%sql
SELECT * FROM bixi_mobility.bronze.trips_raw LIMIT 10;

In [0]:
%sql
-- Nombre total de lignes ingérées
SELECT COUNT(*) FROM bixi_mobility.bronze.trips_raw;

In [0]:
%sql
-- Types de toutes les colonnes
DESCRIBE bixi_mobility.bronze.trips_raw;

In [0]:
%sql
-- Vérification si Auto Loader a corrigé des lignes mal formées (colonne _rescued_data)
SELECT COUNT(*) AS lignes_avec_probleme
FROM bixi_mobility.bronze.trips_raw
WHERE _rescued_data IS NOT NULL;